# NB4 — Fresh Sandbox Generation & Dataset Integration (Rebuilt)

**Purpose:** Generate a fresh, deterministic Python Sandbox dataset; label each action by bounded recovery search at H1/H2/H3; independently audit the resulting labels using a separately implemented reference transition model; integrate the verified NB3 ScienceWorld pilot; and create scenario-grouped, stratified splits.

## Important execution policy

- This notebook creates a **new versioned Sandbox dataset**. It does not read old NB4 Sandbox JSONL or validation reports as inputs.
- The previous NB4 artifacts are retained in timestamped Drive snapshots. They are not deleted.
- The exact NB3 canonical file is imported from the verified NB3 snapshot if it is not present in the current Colab runtime.
- All Sandbox actions are simulated in memory. No real files, emails, accounts, or money are accessed.
- One row represents one scenario variant. H1/H2/H3 are alternative labels for that row, not three independent samples.
- The H3 label is the primary `label` field; all three horizon labels and recovery depths are retained in metadata.

**Run all cells from the top.** The cleanup section is optional and defaults to dry-run only.

**Clean rerun edition:** JSONL writing uses real line breaks; the current NB3 canonical artifact is discovered and SHA-256 cross-checked dynamically; ScienceWorld rows are normalised from the actual NB3 schema; obsolete manual JSONL repair cells have been removed.


## 1. Project setup and run configuration

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, deque, defaultdict
import hashlib, json, os, random, shutil, sys, platform

SEED = 20261119
N_PAIRS = 1000
TARGET_SANDBOX_ROWS = N_PAIRS * 2
HORIZONS = (1, 2, 3)
TEST_FRACTION = 0.20
SPLIT_SEEDS = (0, 1, 2)
SPLIT_METHOD = "StratifiedGroupKFold_v2"

# Fresh generator version: deliberately incompatible with the previous NB4 generator.
GENERATOR_VERSION = "2.0.0"

# Optional cleanup: keep False for the first pass. The cleanup cell only targets
# known NB4 working artifacts under /content/irrev, never Drive snapshots.
RUN_OPTIONAL_NB4_CLEANUP = False

# NB3 canonical source is discovered dynamically in Section 9.
# Do not pin an old timestamped filename here.

PROJECT_ROOT = Path("/content/irrev")
DRIVE_ROOT = Path("/content/drive/MyDrive/ir")
BACKUP_ROOT = Path("/content/drive/MyDrive/ir_final")
DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports"
SANDBOX_DIR = DATA_DIR / "sandbox"
INTEGRATED_DIR = DATA_DIR / "integrated"

for p in [PROJECT_ROOT, DATA_DIR, REPORT_DIR, SANDBOX_DIR, INTEGRATED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Project:", PROJECT_ROOT)
print("Generator:", GENERATOR_VERSION)
print("Seed:", SEED)


Python: 3.13.18
Project: /content/irrev
Generator: 2.0.0
Seed: 20261119


## 2. Mount Drive and define file helpers

In [2]:
# ============================================================
# SECTION 2 — MOUNT DRIVE AND DEFINE FILE HELPERS
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped/unavailable:", repr(e))

for p in [DRIVE_ROOT, BACKUP_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

def write_jsonl(path, rows):
    """Write one JSON object per physical line using a real newline."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, sort_keys=True))
            f.write("\n")

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except Exception as e:
                    raise ValueError(
                        f"Invalid JSONL at {path}:{line_no}: {e}"
                    ) from e
    return rows

print("Drive MyDrive available:", Path("/content/drive/MyDrive").exists())
print("JSONL writer configured with actual newline characters.")


Mounted at /content/drive
Drive MyDrive available: True
JSONL writer configured with actual newline characters.


## 3. Optional cleanup of old NB4 working files

This is deliberately conservative. It lists only the known NB4 working outputs inside `/content/irrev`. It does **not** delete:
- any timestamped Drive snapshot under `/content/drive/MyDrive/ir` or `/content/drive/MyDrive/ir_final`;
- NB1, NB2, or NB3 data;
- the NB3 canonical JSONL;
- the source notebooks.

Leave `RUN_OPTIONAL_NB4_CLEANUP = False` for a dry run. If the printed list is exactly what you intend to remove, change it to `True` and rerun this cell. The fresh NB4 cells below also overwrite their own named working outputs, so cleanup is optional.

In [3]:
NB4_WORKING_FILES = [
    SANDBOX_DIR / "sandbox_pairs.jsonl",
    SANDBOX_DIR / "sandbox_manifest.json",
    REPORT_DIR / "NB4_sandbox_bfs_validation.json",
    INTEGRATED_DIR / "integrated_dataset.jsonl",
    INTEGRATED_DIR / "scienceworld_unlabelled_candidates.jsonl",
    INTEGRATED_DIR / "grouped_splits.json",
    REPORT_DIR / "NB4_dataset_summary.json",
    REPORT_DIR / "NB4_DATA_CARD.md",
    REPORT_DIR / "NB4_reproducibility_manifest.json",
]
NB4_WORKING_FILES += list(REPORT_DIR.glob("NB4_snapshot_*_backup_manifest.json"))

existing_nb4_files = [p for p in NB4_WORKING_FILES if p.exists()]
print("NB4 working files identified:")
for p in existing_nb4_files:
    print(" -", p, "|", p.stat().st_size, "bytes")

if RUN_OPTIONAL_NB4_CLEANUP:
    for p in existing_nb4_files:
        if p.is_file():
            p.unlink()
    print("Deleted", len(existing_nb4_files), "NB4 working files. Drive snapshots were not touched.")
else:
    print("DRY RUN ONLY — no files deleted. Set RUN_OPTIONAL_NB4_CLEANUP=True to delete this exact list.")


NB4 working files identified:
DRY RUN ONLY — no files deleted. Set RUN_OPTIONAL_NB4_CLEANUP=True to delete this exact list.


## 4. Fresh Sandbox transition model

In [4]:
LABELS = ("reversible", "partially_reversible", "irreversible")
FAMILIES = ("edit", "delete", "email", "transfer")

def make_spec(pair_index, family, variant, seed=SEED):
    # Both members of a pair share every field except recovery_enabled.
    # Recovery plan length varies across scenarios to make H1/H2/H3 informative.
    rng = random.Random(seed * 1000003 + pair_index * 97 + variant)
    recovery_steps = 1 + (pair_index % 3)  # 1, 2, or 3 recovery actions
    recovery_enabled = (variant == 0)
    recovery_quality = "exact" if family in ("edit", "delete", "email") else "approximate"
    if family == "transfer" and pair_index % 2 == 0:
        recovery_quality = "exact"
    return {
        "scenario_id": f"sbx-v2-{pair_index:05d}",
        "family": family,
        "verb": {"edit":"edit_file","delete":"delete_file","email":"send_email","transfer":"transfer_money"}[family],
        "target": f"object-{pair_index:05d}",
        "original_value": f"original-{pair_index:05d}",
        "changed_value": f"changed-{pair_index:05d}",
        "amount": 100 + (pair_index % 5) * 25,
        "recovery_enabled": recovery_enabled,
        "recovery_steps": recovery_steps,
        "recovery_quality": recovery_quality,
    }

def initial_state(spec):
    return {
        "scenario_id": spec["scenario_id"],
        "family": spec["family"],
        "value": spec["original_value"],
        "balance": 1000,
        "deleted": False,
        "sent": False,
        "transfer_settled": False,
        "recovery_stage": 0,
        "recovery_enabled": spec["recovery_enabled"],
        "recovery_steps": spec["recovery_steps"],
        "recovery_quality": spec["recovery_quality"],
        "intervention_applied": False,
        "audit_trail": [],
    }

def canonical_state(state):
    return json.dumps(state, sort_keys=True, separators=(",", ":"))

def apply_intervention(state, spec):
    if state["intervention_applied"]:
        return None
    s = json.loads(json.dumps(state))
    family = spec["family"]
    if family == "edit":
        s["value"] = spec["changed_value"]
    elif family == "delete":
        s["deleted"] = True
    elif family == "email":
        s["sent"] = True
    elif family == "transfer":
        s["balance"] -= spec["amount"]
        s["transfer_settled"] = True
    else:
        raise ValueError(family)
    s["intervention_applied"] = True
    s["audit_trail"].append("intervention")
    return s

def available_recovery_actions(state, spec):
    if not state["intervention_applied"] or not state["recovery_enabled"]:
        return []
    if state["recovery_stage"] >= spec["recovery_steps"]:
        return []
    # Family-specific named stages, with one legal next action per stage.
    stage = state["recovery_stage"] + 1
    names = {
        "edit": ["locate_version", "restore_content", "restore_metadata"],
        "delete": ["locate_deleted_item", "restore_object", "restore_metadata"],
        "email": ["request_recall", "retract_delivery", "restore_pre_send_record"],
        "transfer": ["open_dispute", "issue_refund", "reconcile_ledger"],
    }
    return [{"kind": names[spec["family"]][stage - 1]}]

def apply_recovery_action(state, action, spec):
    s = json.loads(json.dumps(state))
    expected = available_recovery_actions(state, spec)
    if not expected or action != expected[0]:
        return None
    s["recovery_stage"] += 1
    s["audit_trail"].append(action["kind"])
    if s["recovery_stage"] == spec["recovery_steps"]:
        # Restore the task-relevant target. Exact restoration also restores audit
        # metadata; approximate restoration leaves a persistent audit trace.
        if spec["family"] == "edit":
            s["value"] = spec["original_value"]
        elif spec["family"] == "delete":
            s["deleted"] = False
        elif spec["family"] == "email":
            s["sent"] = False
        elif spec["family"] == "transfer":
            s["balance"] += spec["amount"]
            s["transfer_settled"] = False
        if spec["recovery_quality"] == "exact":
            s["audit_trail"] = []
            s["intervention_applied"] = False
            s["recovery_stage"] = 0
    return s

def exact_recovery(state, baseline):
    return state == baseline

def approximate_recovery(state, baseline):
    # Operational definition: target-level effect is restored, but the persistent
    # audit trail/intervention marker means the full state is not identical.
    keys = ("value", "balance", "deleted", "sent", "transfer_settled")
    return all(state[k] == baseline[k] for k in keys) and state != baseline


## 5. Bounded BFS labeler (H1/H2/H3)

The BFS starts from the post-intervention state and explores only legal recovery actions supplied by the transition model. It records the minimum exact and approximate recovery depths. Labels are horizon-specific:
- `reversible`: exact baseline state recovered within the horizon;
- `partially_reversible`: task-relevant target state recovered, but complete state is not identical;
- `irreversible`: neither recovery criterion is reached within the horizon.

This is explicitly **bounded recoverability in this simulated action space**, not a claim of absolute irreversibility.

In [5]:
def bfs_recovery_depths(spec, horizon):
    baseline = initial_state(spec)
    post = apply_intervention(baseline, spec)
    if post is None:
        raise RuntimeError("Intervention failed")

    queue = deque([(post, 0)])
    seen = {canonical_state(post)}
    exact_depth = None
    approximate_depth = None

    while queue:
        state, depth = queue.popleft()
        if depth > 0:
            if exact_recovery(state, baseline):
                exact_depth = depth if exact_depth is None else min(exact_depth, depth)
            elif approximate_recovery(state, baseline):
                approximate_depth = depth if approximate_depth is None else min(approximate_depth, depth)

        if depth >= horizon:
            continue
        for action in available_recovery_actions(state, spec):
            nxt = apply_recovery_action(state, action, spec)
            if nxt is None:
                continue
            key = canonical_state(nxt)
            if key not in seen:
                seen.add(key)
                queue.append((nxt, depth + 1))

    return {"exact_depth": exact_depth, "approximate_depth": approximate_depth}

def label_from_depths(depths, horizon):
    if depths["exact_depth"] is not None and depths["exact_depth"] <= horizon:
        return "reversible"
    if depths["approximate_depth"] is not None and depths["approximate_depth"] <= horizon:
        return "partially_reversible"
    return "irreversible"

# Sanity cases: recovery depths 1, 2, and 3 must change the horizon label as expected.
for depth in (1, 2, 3):
    spec = make_spec(depth, "edit", 0)
    spec["recovery_steps"] = depth
    got = [label_from_depths(bfs_recovery_depths(spec, h), h) for h in HORIZONS]
    expected = ["reversible" if h >= depth else "irreversible" for h in HORIZONS]
    assert got == expected, (depth, got, expected)
print("BFS horizon sanity checks passed.")


BFS horizon sanity checks passed.


## 6. Generate fresh minimal-pair Sandbox dataset

In [6]:
def make_state_text(spec):
    # Expose realistic family-specific controls, not internal labels, recovery depth,
    # or the generic recovery_enabled/recovery_quality variables.
    family = spec["family"]
    if family == "edit":
        controls = f"version_history_available={spec['recovery_enabled']}"
    elif family == "delete":
        controls = f"recycle_bin_enabled={spec['recovery_enabled']}"
    elif family == "email":
        controls = f"recall_window_open={spec['recovery_enabled']}"
    elif family == "transfer":
        controls = f"refund_eligible={spec['recovery_enabled']}; transfer_status=settled"
    else:
        raise ValueError(family)
    return (
        f"Simulated {family} operation. Target={spec['target']}; "
        f"original_value={spec['original_value']}; changed_value={spec['changed_value']}; "
        f"amount={spec['amount']}; {controls}."
    )

def generate_sandbox_rows(n_pairs=N_PAIRS, seed=SEED):
    rows = []
    for i in range(n_pairs):
        family = FAMILIES[i % len(FAMILIES)]
        for variant in (0, 1):
            spec = make_spec(i, family, variant, seed)
            horizon_labels = {}
            depths_by_horizon = {}
            for h in HORIZONS:
                depths = bfs_recovery_depths(spec, h)
                depths_by_horizon[str(h)] = depths
                horizon_labels[str(h)] = label_from_depths(depths, h)
            action_text = {
                "edit": f"Edit {spec['target']} to {spec['changed_value']}",
                "delete": f"Delete {spec['target']}",
                "email": f"Send simulated email for {spec['target']}",
                "transfer": f"Transfer simulated amount {spec['amount']}",
            }[family]
            rows.append({
                "id": f"sbx-v2-row-{i:05d}-{variant}",
                "environment": "python_sandbox",
                "state_text": make_state_text(spec),
                "action_text": action_text,
                "label": horizon_labels["3"],
                "horizon": 3,
                "source": "fresh_simulated_sandbox_bfs_v2",
                "meta": {
                    "scenario_id": spec["scenario_id"],
                    "pair_id": spec["scenario_id"],
                    "pair_variant": variant,
                    "family": family,
                    "verb": spec["verb"],
                    "spec": spec,
                    "labels_by_horizon": horizon_labels,
                    "recovery_depths_by_horizon": depths_by_horizon,
                    "label_scope": "bounded recovery in the declared simulated action space",
                },
            })
    return rows

SANDBOX_JSONL = SANDBOX_DIR / "sandbox_pairs_v2.jsonl"
SANDBOX_MANIFEST = SANDBOX_DIR / "sandbox_manifest_v2.json"
GEN_CONFIG = {
    "seed": SEED, "n_pairs": N_PAIRS, "horizons": list(HORIZONS),
    "families": list(FAMILIES), "generator_version": GENERATOR_VERSION,
    "recovery_plan": "one legal staged action per BFS depth; variant 0 enabled, variant 1 disabled",
}
GEN_CONFIG_HASH = hashlib.sha256(json.dumps(GEN_CONFIG, sort_keys=True).encode()).hexdigest()

sandbox_rows = generate_sandbox_rows()
write_jsonl(SANDBOX_JSONL, sandbox_rows)
sandbox_hash = sha256_file(SANDBOX_JSONL)
write_json(SANDBOX_MANIFEST, {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "config": GEN_CONFIG, "config_hash": GEN_CONFIG_HASH,
    "record_count": len(sandbox_rows), "dataset_sha256": sandbox_hash,
    "generator": "NB4 rebuilt staged-recovery Sandbox",
})
print("Generated:", len(sandbox_rows), "rows")
print("Class counts at H3:", dict(Counter(r["label"] for r in sandbox_rows)))
print("Horizon class counts:")
for h in HORIZONS:
    print("H", h, dict(Counter(r["meta"]["labels_by_horizon"][str(h)] for r in sandbox_rows)))
print("SHA-256:", sandbox_hash)
assert len(sandbox_rows) == TARGET_SANDBOX_ROWS


Generated: 2000 rows
Class counts at H3: {'reversible': 750, 'irreversible': 1000, 'partially_reversible': 250}
Horizon class counts:
H 1 {'reversible': 250, 'irreversible': 1666, 'partially_reversible': 84}
H 2 {'reversible': 500, 'irreversible': 1333, 'partially_reversible': 167}
H 3 {'reversible': 750, 'irreversible': 1000, 'partially_reversible': 250}
SHA-256: 08726639568dc289313e00707349cea7a270e513a379e4369de7ec1959a2a1d8


## 7. Independent audit using a separate reference transition implementation

The audit below does not call the generator's `apply_intervention`, `available_recovery_actions`, `apply_recovery_action`, `bfs_recovery_depths`, or `label_from_depths`. It reconstructs the expected reachable depth from a separate declarative reference model, then compares all H1/H2/H3 labels and depth records. This is a second implementation of the declared transition semantics, not an external empirical ground truth.

In [7]:
def reference_recovery_depth(spec):
    # Independent declarative oracle: no generator transition functions used.
    if not spec["recovery_enabled"]:
        return None, None
    steps = int(spec["recovery_steps"])
    if spec["recovery_quality"] == "exact":
        return steps, None
    if spec["recovery_quality"] == "approximate":
        return None, steps
    raise ValueError("Unknown recovery quality")

def reference_label(spec, horizon):
    exact_depth, approx_depth = reference_recovery_depth(spec)
    if exact_depth is not None and exact_depth <= horizon:
        return "reversible"
    if approx_depth is not None and approx_depth <= horizon:
        return "partially_reversible"
    return "irreversible"

audit_failures = []
for row in sandbox_rows:
    spec = row["meta"]["spec"]
    for h in HORIZONS:
        expected = reference_label(spec, h)
        actual = row["meta"]["labels_by_horizon"][str(h)]
        if actual != expected:
            audit_failures.append({
                "id": row["id"], "horizon": h,
                "stored": actual, "reference": expected,
            })
    if row["label"] != reference_label(spec, 3):
        audit_failures.append({"id": row["id"], "field": "primary_H3_label"})

assert not audit_failures, audit_failures[:10]
print("Independent reference audit passed for", len(sandbox_rows), "rows ×", len(HORIZONS), "horizons.")


Independent reference audit passed for 2000 rows × 3 horizons.


## 8. Structural, pair, and dataset validation

In [8]:
def validate_sandbox(rows):
    errors = []
    required = {"id","environment","state_text","action_text","label","horizon","source","meta"}
    ids = set()
    for r in rows:
        missing = required - set(r)
        if missing:
            errors.append(f"{r.get('id')}: missing {sorted(missing)}")
            continue
        if r["id"] in ids:
            errors.append(f"duplicate_id:{r['id']}")
        ids.add(r["id"])
        if r["label"] not in LABELS:
            errors.append(f"invalid_label:{r['id']}")
        if not r["meta"].get("scenario_id") or not r["meta"].get("pair_id"):
            errors.append(f"missing_pair_or_scenario:{r['id']}")
        for h in HORIZONS:
            if r["meta"]["labels_by_horizon"].get(str(h)) not in LABELS:
                errors.append(f"invalid_horizon_label:{r['id']}:{h}")
            if r["label"] != r["meta"]["labels_by_horizon"]["3"]:
                errors.append(f"H3_primary_mismatch:{r['id']}")
    return errors

pair_groups = defaultdict(list)
for row in sandbox_rows:
    pair_groups[row["meta"]["pair_id"]].append(row)

pair_failures = []
for pair_id, members in pair_groups.items():
    if len(members) != 2:
        pair_failures.append({"pair_id": pair_id, "count": len(members)})
        continue
    a, b = sorted(members, key=lambda x: x["meta"]["pair_variant"])
    sa, sb = a["meta"]["spec"], b["meta"]["spec"]
    differing = [k for k in sa if sa[k] != sb[k]]
    if differing != ["recovery_enabled"]:
        pair_failures.append({"pair_id": pair_id, "differing_spec_fields": differing})
    if a["meta"]["verb"] != b["meta"]["verb"]:
        pair_failures.append({"pair_id": pair_id, "verb_mismatch": True})

sandbox_errors = validate_sandbox(sandbox_rows)
assert not sandbox_errors, sandbox_errors[:20]
assert not pair_failures, pair_failures[:20]
assert len(pair_groups) == N_PAIRS
print("Schema errors:", len(sandbox_errors))
print("Pair failures:", len(pair_failures))
print("Valid minimal pairs:", len(pair_groups))


Schema errors: 0
Pair failures: 0
Valid minimal pairs: 1000


## 9. Recover and import the exact NB3 ScienceWorld pilot

In [9]:
from pathlib import Path
from collections import Counter

configured_sw = (
    PROJECT_ROOT / "data/scienceworld/processed"
)

snapshot_roots = [
    DRIVE_ROOT,
    BACKUP_ROOT,
]

candidate_paths = []

# Search the current runtime for canonical NB3 files.
if configured_sw.exists():
    candidate_paths.extend(
        configured_sw.glob("scienceworld_pilot_canonical_*.jsonl")
    )

# Discover timestamped NB3 snapshots in both Drive roots.
for root in snapshot_roots:
    if not root.exists():
        continue

    snapshots = sorted(
        root.glob("NB3_snapshot_*"),
        key=lambda p: p.name,
        reverse=True,
    )

    for snapshot in snapshots:
        processed_dir = (
            snapshot / "data/scienceworld/processed"
        )

        if processed_dir.exists():
            candidate_paths.extend(
                processed_dir.glob(
                    "scienceworld_pilot_canonical_*.jsonl"
                )
            )

# Remove duplicate paths while preserving order.
candidate_paths = list(dict.fromkeys(candidate_paths))

if not candidate_paths:
    raise FileNotFoundError(
        "No NB3 canonical ScienceWorld JSONL found in the "
        "current runtime or either Drive root."
    )

# Select the newest canonical artifact by filename timestamp.
candidate_paths.sort(
    key=lambda p: p.name,
    reverse=True,
)

scienceworld_path = candidate_paths[0]
selected_hash = sha256_file(scienceworld_path)

# Verify every available copy of this exact artifact.
matching_copies = [
    p for p in candidate_paths
    if p.name == scienceworld_path.name
]

if any(sha256_file(p) != selected_hash for p in matching_copies):
    raise RuntimeError(
        "Copies of the selected NB3 canonical JSONL disagree by SHA-256."
    )

scienceworld_raw = read_jsonl(scienceworld_path)

if not scienceworld_raw:
    raise ValueError("NB3 canonical JSONL is empty.")

print("NB3 source:", scienceworld_path)
print("NB3 records:", len(scienceworld_raw))
print("NB3 SHA-256:", selected_hash)
print("Verified matching copies:", len(matching_copies))
print(
    "NB3 labels:",
    dict(Counter(r.get("label", "MISSING") for r in scienceworld_raw))
)

NB3 source: /content/drive/MyDrive/ir/NB3_snapshot_20261123T123703377799Z/data/scienceworld/processed/scienceworld_pilot_canonical_20261123T123703338645Z.jsonl
NB3 records: 2
NB3 SHA-256: 4922ea13834b3090fae04cc1f6777c13c3c934c5d2ad20909700d199c9c707be
Verified matching copies: 2
NB3 labels: {'reversible': 1, 'unlabelled_candidate': 1}


In [10]:
# ============================================================
# SECTION 9 — NORMALISE NB3 SCIENCEWORLD RECORDS
# ============================================================

def describe_initial_target(record):
    objects = record.get("target_objects_before", [])
    if not objects:
        return (
            f"Target object: {record.get('target_object', 'unspecified')}. "
            "No initial target-object observation was recorded."
        )

    descriptions = []
    for obj in objects:
        if not isinstance(obj, dict):
            descriptions.append(str(obj))
            continue

        fields = [
            f"UUID={obj.get('uuid', 'unknown')}",
            f"name={obj.get('name', obj.get('object', 'unknown'))}",
            f"path={obj.get('path', 'unknown')}",
        ]
        for key in (
            "type", "material", "colour", "color", "state",
            "temperatureC", "edible", "poisonous"
        ):
            if key in obj:
                fields.append(f"{key}={obj[key]}")
        descriptions.append("; ".join(fields))

    return "Initial target observation: " + " | ".join(descriptions)


def normalize_scienceworld_row(record, index):
    record_id = record.get("record_id")
    if not record_id:
        raise ValueError(f"NB3 record {index} has no record_id.")

    original_label = record.get("label", "unlabelled_candidate")
    label = original_label if original_label in LABELS else "unlabelled_candidate"

    scenario_name = record.get("scenario_name", f"scienceworld_pilot_{index:03d}")
    scenario_id = f"scienceworld::{scenario_name}"
    task = record.get("task", "unspecified")
    variation = record.get("variation", "unspecified")
    target = record.get("target_object", "unspecified")
    init_action = record.get("initialisation_action", "")

    state_parts = [
        f"ScienceWorld task: {task}.",
        f"Variation: {variation}.",
        f"Target object: {target}.",
        describe_initial_target(record),
    ]
    if init_action:
        state_parts.append(f"Environment initialisation observation/action: {init_action}.")

    action_sequence = record.get("action_sequence", [])
    if not isinstance(action_sequence, list):
        raise TypeError(f"NB3 action_sequence must be a list: {record_id}")
    action_text = " ; ".join(str(a) for a in action_sequence)
    if not action_text:
        action_text = "No intervention action was recorded."

    try:
        horizon = int(record.get("horizon", 3))
    except (TypeError, ValueError):
        horizon = 3

    meta = {
        "scenario_id": scenario_id,
        "pair_id": None,
        "scenario_name": scenario_name,
        "task": task,
        "variation": variation,
        "target_object": target,
        "label_scope": record.get("label_scope"),
        "label_confidence": record.get("label_confidence"),
        "evidence_scope": (
            "ScienceWorld pilot observation; preserve the original NB3 scope "
            "and do not generalise beyond its evidence."
        ),
        "structural_recovered": record.get("structural_recovered"),
        "full_state_recovered": record.get("full_state_recovered"),
        "initial_structural_fingerprint": record.get("initial_structural_fingerprint"),
        "final_structural_fingerprint": record.get("final_structural_fingerprint"),
        "action_sequence": action_sequence,
        "recovery_sequence": record.get("recovery_sequence", []),
        "target_uuids_missing_after": record.get("target_uuids_missing_after", []),
        "provenance": record.get("provenance", {}),
        "original_record": record,
    }

    return {
        "id": f"scienceworld::{record_id}",
        "environment": "scienceworld",
        "state_text": " ".join(state_parts),
        "action_text": action_text,
        "label": label,
        "horizon": horizon,
        "source": "NB3_scienceworld_pilot",
        "meta": meta,
    }

normalized_sw = [
    normalize_scienceworld_row(record, i)
    for i, record in enumerate(scienceworld_raw)
]
eligible_sw = [row for row in normalized_sw if row["label"] in LABELS]
unlabelled_sw = [row for row in normalized_sw if row["label"] == "unlabelled_candidate"]

# Validate the known, currently authoritative NB3 pilot snapshot.
assert len(normalized_sw) == 2, f"Expected 2 NB3 pilot records, found {len(normalized_sw)}"
assert len(eligible_sw) == 1, f"Expected 1 labelled pilot, found {len(eligible_sw)}"
assert len(unlabelled_sw) == 1, f"Expected 1 unlabelled candidate, found {len(unlabelled_sw)}"

assert eligible_sw[0]["id"] == "scienceworld::84210c698161f05e"
assert eligible_sw[0]["label"] == "reversible"
assert eligible_sw[0]["meta"]["structural_recovered"] is True
assert eligible_sw[0]["meta"]["full_state_recovered"] is False

assert unlabelled_sw[0]["id"] == "scienceworld::37980b29a18336cb"
assert unlabelled_sw[0]["label"] == "unlabelled_candidate"
assert unlabelled_sw[0]["meta"]["target_uuids_missing_after"] == ["17598"]

print("NB3 records:", len(normalized_sw))
print("ScienceWorld labelled:", len(eligible_sw))
print("ScienceWorld unlabelled candidates:", len(unlabelled_sw))
print("Labelled pilot:", eligible_sw[0]["id"], eligible_sw[0]["label"])
print("Unlabelled candidate:", unlabelled_sw[0]["id"], unlabelled_sw[0]["label"])
print("Original NB3 evidence preserved; orange candidate remains unlabelled.")


NB3 records: 2
ScienceWorld labelled: 1
ScienceWorld unlabelled candidates: 1
Labelled pilot: scienceworld::84210c698161f05e reversible
Unlabelled candidate: scienceworld::37980b29a18336cb unlabelled_candidate
Original NB3 evidence preserved; orange candidate remains unlabelled.


## 10. Integrate labelled rows and preserve unlabelled candidates

In [11]:
# ============================================================
# SECTION 10 — INTEGRATE LABELLED ROWS AND PRESERVE CANDIDATES
# ============================================================

SANDBOX_JSONL = SANDBOX_DIR / "sandbox_pairs_v2.jsonl"
SANDBOX_MANIFEST = SANDBOX_DIR / "sandbox_manifest_v2.json"
UNLABELLED_JSONL = INTEGRATED_DIR / "scienceworld_unlabelled_candidates_v2.jsonl"
INTEGRATED_JSONL = INTEGRATED_DIR / "integrated_dataset_v2.jsonl"

# Re-read Sandbox from disk so integration is based on the persisted artifact.
disk_sandbox_source = read_jsonl(SANDBOX_JSONL)
assert len(disk_sandbox_source) == TARGET_SANDBOX_ROWS
assert len({r["id"] for r in disk_sandbox_source}) == TARGET_SANDBOX_ROWS

integrated_rows = []
seen_ids = set()
for row in disk_sandbox_source + eligible_sw:
    item = json.loads(json.dumps(row))
    if item["id"] in seen_ids:
        raise ValueError(f"Duplicate ID during integration: {item['id']}")
    seen_ids.add(item["id"])
    integrated_rows.append(item)


def validate_integrated(rows):
    errors = []
    required = {
        "id", "environment", "state_text", "action_text",
        "label", "horizon", "source", "meta"
    }
    ids = set()
    for row in rows:
        missing = required - set(row)
        if missing:
            errors.append(f"{row.get('id', 'UNKNOWN')}: missing {sorted(missing)}")
            continue
        if row["id"] in ids:
            errors.append(f"duplicate_id:{row['id']}")
        ids.add(row["id"])
        if row["label"] not in LABELS:
            errors.append(f"invalid_label:{row['id']}")
        if not row.get("meta", {}).get("scenario_id"):
            errors.append(f"missing_scenario_id:{row['id']}")
        if not str(row["state_text"]).strip():
            errors.append(f"empty_state_text:{row['id']}")
        if not str(row["action_text"]).strip():
            errors.append(f"empty_action_text:{row['id']}")
    return errors

integrated_errors = validate_integrated(integrated_rows)
assert not integrated_errors, integrated_errors[:20]
assert len(integrated_rows) == 2001
assert len(eligible_sw) == 1 and len(unlabelled_sw) == 1

write_jsonl(UNLABELLED_JSONL, unlabelled_sw)
write_jsonl(INTEGRATED_JSONL, integrated_rows)

# Verify exact disk round-trip.
disk_integrated_check = read_jsonl(INTEGRATED_JSONL)
disk_unlabelled_check = read_jsonl(UNLABELLED_JSONL)
assert len(disk_integrated_check) == 2001
assert len(disk_unlabelled_check) == 1
assert [r["id"] for r in disk_integrated_check] == [r["id"] for r in integrated_rows]
assert disk_integrated_check == integrated_rows
assert disk_unlabelled_check[0]["label"] == "unlabelled_candidate"

print("Sandbox rows:", len(disk_sandbox_source))
print("ScienceWorld labelled rows:", len(eligible_sw))
print("Integrated labelled rows:", len(disk_integrated_check))
print("Unlabelled candidates kept separately:", len(disk_unlabelled_check))
print("Integrated classes:", dict(Counter(r["label"] for r in disk_integrated_check)))
print("Integrated SHA-256:", sha256_file(INTEGRATED_JSONL))


Sandbox rows: 2000
ScienceWorld labelled rows: 1
Integrated labelled rows: 2001
Unlabelled candidates kept separately: 1
Integrated classes: {'reversible': 751, 'irreversible': 1000, 'partially_reversible': 250}
Integrated SHA-256: 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd


## 11. Stratified scenario-grouped splits with explicit leakage checks

In [12]:
# ============================================================
# SECTION 11 — STRATIFIED SCENARIO-GROUPED SPLITS
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold


def make_grouped_splits(rows):
    y = [row["label"] for row in rows]
    groups = [
        f'{row["environment"]}::{row["meta"]["scenario_id"]}'
        for row in rows
    ]
    n_splits = round(1 / TEST_FRACTION)
    output = {}

    for seed in SPLIT_SEEDS:
        splitter = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed
        )
        train_idx, test_idx = next(splitter.split(rows, y, groups))
        output[str(seed)] = {
            "train_ids": sorted(rows[i]["id"] for i in train_idx),
            "test_ids": sorted(rows[i]["id"] for i in test_idx),
        }

    return {
        "dataset_sha256": sha256_file(INTEGRATED_JSONL),
        "split_method": SPLIT_METHOD,
        "test_fraction": TEST_FRACTION,
        "seeds": list(SPLIT_SEEDS),
        "splits": output,
    }

splits_obj = make_grouped_splits(integrated_rows)
splits = splits_obj["splits"]
by_id = {row["id"]: row for row in integrated_rows}
split_diagnostics = {}

for seed, split in splits.items():
    train_ids = set(split["train_ids"])
    test_ids = set(split["test_ids"])
    assert train_ids.isdisjoint(test_ids), f"Record-ID leakage in seed {seed}"
    assert train_ids | test_ids == set(by_id), f"Coverage mismatch in seed {seed}"

    train_groups = {
        (by_id[x]["environment"], by_id[x]["meta"]["scenario_id"])
        for x in train_ids
    }
    test_groups = {
        (by_id[x]["environment"], by_id[x]["meta"]["scenario_id"])
        for x in test_ids
    }
    assert train_groups.isdisjoint(test_groups), f"Scenario leakage in seed {seed}"

    split_diagnostics[seed] = {}
    for part, ids in (("train", train_ids), ("test", test_ids)):
        selected = [by_id[x] for x in ids]
        split_diagnostics[seed][part] = {
            "records": len(selected),
            "class_counts": dict(Counter(row["label"] for row in selected)),
            "environment_counts": dict(Counter(row["environment"] for row in selected)),
        }

SPLITS_JSON = INTEGRATED_DIR / "grouped_splits_v2.json"
write_json(SPLITS_JSON, splits_obj)

# Read-back check and verify the dataset hash is bound to this split artifact.
saved_splits_obj = read_json(SPLITS_JSON)
assert saved_splits_obj["dataset_sha256"] == sha256_file(INTEGRATED_JSONL)
assert set(saved_splits_obj["splits"]) == {str(seed) for seed in SPLIT_SEEDS}

print("Split method:", SPLIT_METHOD)
print("Split seeds:", list(splits.keys()))
print("Dataset hash recorded:", splits_obj["dataset_sha256"])
print(json.dumps(split_diagnostics, indent=2))
print("All grouped split checks passed.")


Split method: StratifiedGroupKFold_v2
Split seeds: ['0', '1', '2']
Dataset hash recorded: 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd
{
  "0": {
    "train": {
      "records": 1599,
      "class_counts": {
        "irreversible": 799,
        "reversible": 596,
        "partially_reversible": 204
      },
      "environment_counts": {
        "python_sandbox": 1598,
        "scienceworld": 1
      }
    },
    "test": {
      "records": 402,
      "class_counts": {
        "partially_reversible": 46,
        "irreversible": 201,
        "reversible": 155
      },
      "environment_counts": {
        "python_sandbox": 402
      }
    }
  },
  "1": {
    "train": {
      "records": 1601,
      "class_counts": {
        "irreversible": 800,
        "reversible": 595,
        "partially_reversible": 206
      },
      "environment_counts": {
        "python_sandbox": 1600,
        "scienceworld": 1
      }
    },
    "test": {
      "records": 400,
      "class_count

## 12. Horizon-aware summaries and data card

In [13]:
# ============================================================
# SECTION 12 — HORIZON-AWARE SUMMARY AND DATA CARD
# ============================================================

horizon_counts = {
    str(h): dict(
        Counter(row["meta"]["labels_by_horizon"][str(h)] for row in sandbox_rows)
    )
    for h in HORIZONS
}

summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "generator_version": GENERATOR_VERSION,
    "seed": SEED,
    "sandbox_records": len(sandbox_rows),
    "integrated_records": len(integrated_rows),
    "unlabelled_candidate_records": len(unlabelled_sw),
    "environment_counts": dict(Counter(row["environment"] for row in integrated_rows)),
    "primary_H3_class_counts": dict(Counter(row["label"] for row in integrated_rows)),
    "sandbox_horizon_class_counts": horizon_counts,
    "scenario_count": len({(row["environment"], row["meta"]["scenario_id"]) for row in integrated_rows}),
    "minimal_pair_count": len(pair_groups),
    "sandbox_sha256": sha256_file(SANDBOX_JSONL),
    "integrated_sha256": sha256_file(INTEGRATED_JSONL),
    "splits_sha256": sha256_file(SPLITS_JSON),
    "unlabelled_candidates_sha256": sha256_file(UNLABELLED_JSONL),
    "nb3_source": str(scienceworld_path),
    "nb3_source_sha256": sha256_file(scienceworld_path),
    "scienceworld_labelled_records": len(eligible_sw),
    "scienceworld_unlabelled_candidates": len(unlabelled_sw),
    "split_method": SPLIT_METHOD,
    "split_diagnostics": split_diagnostics,
    "limitations": [
        "The Python Sandbox is a deterministic, rule-based in-memory simulation, not a real operating environment.",
        "Labels mean bounded recoverability within the declared action space and horizon, not proof of absolute irreversibility.",
        "The single labelled ScienceWorld pilot record is insufficient for ScienceWorld-only test metrics or cross-environment generalisation.",
        "The orange-consumption ScienceWorld record remains an unlabelled candidate because bounded recovery was not tested.",
        "H1/H2/H3 labels are repeated horizon views of the same scenarios, not independent samples.",
    ],
}

SUMMARY_JSON = REPORT_DIR / "NB4_dataset_summary_v2.json"
DATA_CARD = REPORT_DIR / "NB4_DATA_CARD_v2.md"
write_json(SUMMARY_JSON, summary)

DATA_CARD.write_text(
    "# NB4 Dataset Data Card (v2)\n\n"
    f"- Generated: {summary['created_utc']}\n"
    f"- Sandbox records: {summary['sandbox_records']}\n"
    f"- Integrated labelled records: {summary['integrated_records']}\n"
    f"- ScienceWorld labelled records: {summary['scienceworld_labelled_records']}\n"
    f"- Unlabelled candidates kept separately: {summary['scienceworld_unlabelled_candidates']}\n"
    f"- Primary H3 classes: {json.dumps(summary['primary_H3_class_counts'])}\n"
    f"- Horizon class counts: {json.dumps(horizon_counts)}\n"
    f"- Minimal pairs: {summary['minimal_pair_count']}\n"
    f"- Split method: {SPLIT_METHOD}\n\n"
    "## Label definition\n"
    "- reversible: exact baseline state recovered within the tested horizon.\n"
    "- partially_reversible: task-relevant target state recovered but full state differs.\n"
    "- irreversible: neither recovery criterion is reached within the tested horizon.\n\n"
    "## ScienceWorld pilot handling\n"
    "- Picture relocation is retained as a labelled reversible pilot with object-level structural recovery scope; full state was not recovered.\n"
    "- Orange consumption remains an unlabelled candidate because bounded recovery was not tested.\n\n"
    "## Scope and limitations\n"
    "- All Python Sandbox operations are simulated in memory.\n"
    "- Labels express bounded recoverability in the declared action space, not absolute irreversibility.\n"
    "- H1/H2/H3 are alternative targets for the same scenarios, not independent samples.\n"
    "- The current ScienceWorld contribution is a tiny pilot and cannot support cross-environment generalisation claims.\n",
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("Summary:", SUMMARY_JSON)
print("Data card:", DATA_CARD)


{
  "created_utc": "2026-11-23T13:00:07.092821+00:00",
  "generator_version": "2.0.0",
  "seed": 20261119,
  "sandbox_records": 2000,
  "integrated_records": 2001,
  "unlabelled_candidate_records": 1,
  "environment_counts": {
    "python_sandbox": 2000,
    "scienceworld": 1
  },
  "primary_H3_class_counts": {
    "reversible": 751,
    "irreversible": 1000,
    "partially_reversible": 250
  },
  "sandbox_horizon_class_counts": {
    "1": {
      "reversible": 250,
      "irreversible": 1666,
      "partially_reversible": 84
    },
    "2": {
      "reversible": 500,
      "irreversible": 1333,
      "partially_reversible": 167
    },
    "3": {
      "reversible": 750,
      "irreversible": 1000,
      "partially_reversible": 250
    }
  },
  "scenario_count": 1001,
  "minimal_pair_count": 1000,
  "sandbox_sha256": "08726639568dc289313e00707349cea7a270e513a379e4369de7ec1959a2a1d8",
  "integrated_sha256": "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd",
  "splits_sh

## 13. Final integrity checks

In [14]:
# ============================================================
# SECTION 13 — FINAL DISK INTEGRITY AND METADATA VALIDATION
# ============================================================

# Read every authoritative artifact from disk using the strict JSONL reader.
disk_sandbox = read_jsonl(SANDBOX_JSONL)
disk_integrated = read_jsonl(INTEGRATED_JSONL)
disk_unlabelled = read_jsonl(UNLABELLED_JSONL)
disk_splits_obj = read_json(SPLITS_JSON)

assert len(disk_sandbox) == TARGET_SANDBOX_ROWS == 2000
assert len(disk_integrated) == 2001
assert len(disk_unlabelled) == 1

sandbox_ids = [row["id"] for row in disk_sandbox]
integrated_ids = [row["id"] for row in disk_integrated]
assert len(sandbox_ids) == len(set(sandbox_ids))
assert len(integrated_ids) == len(set(integrated_ids))
assert disk_integrated == integrated_rows

# Composition and labels.
assert Counter(row["environment"] for row in disk_integrated) == Counter({
    "python_sandbox": 2000,
    "scienceworld": 1,
})
assert Counter(row["label"] for row in disk_integrated) == Counter({
    "irreversible": 1000,
    "reversible": 751,
    "partially_reversible": 250,
})

sw_labelled = [row for row in disk_integrated if row["environment"] == "scienceworld"]
assert len(sw_labelled) == 1
assert sw_labelled[0]["id"] == "scienceworld::84210c698161f05e"
assert sw_labelled[0]["label"] == "reversible"
assert sw_labelled[0]["meta"]["structural_recovered"] is True
assert sw_labelled[0]["meta"]["full_state_recovered"] is False

assert disk_unlabelled[0]["id"] == "scienceworld::37980b29a18336cb"
assert disk_unlabelled[0]["label"] == "unlabelled_candidate"
assert disk_unlabelled[0]["id"] not in set(integrated_ids)

# Revalidate all saved grouped splits.
disk_by_id = {row["id"]: row for row in disk_integrated}
all_integrated_ids = set(integrated_ids)
assert disk_splits_obj["dataset_sha256"] == sha256_file(INTEGRATED_JSONL)
for seed, split in disk_splits_obj["splits"].items():
    train_ids = set(split["train_ids"])
    test_ids = set(split["test_ids"])
    assert train_ids.isdisjoint(test_ids), f"Record leakage in seed {seed}"
    assert train_ids | test_ids == all_integrated_ids, f"Coverage mismatch in seed {seed}"
    train_groups = {
        (disk_by_id[x]["environment"], disk_by_id[x]["meta"]["scenario_id"])
        for x in train_ids
    }
    test_groups = {
        (disk_by_id[x]["environment"], disk_by_id[x]["meta"]["scenario_id"])
        for x in test_ids
    }
    assert train_groups.isdisjoint(test_groups), f"Scenario leakage in seed {seed}"

# Independent Sandbox validation and minimal-pair audit must still pass.
assert len(audit_failures) == 0
assert len(pair_failures) == 0
assert len(sandbox_errors) == 0
assert len(integrated_errors) == 0
assert len(pair_groups) == N_PAIRS

# Refresh metadata hashes only after all data checks have passed.
sandbox_hash = sha256_file(SANDBOX_JSONL)
integrated_hash = sha256_file(INTEGRATED_JSONL)
splits_hash = sha256_file(SPLITS_JSON)
unlabelled_hash = sha256_file(UNLABELLED_JSONL)

sandbox_manifest_obj = read_json(SANDBOX_MANIFEST)
sandbox_manifest_obj["record_count"] = len(disk_sandbox)
sandbox_manifest_obj["dataset_sha256"] = sandbox_hash
sandbox_manifest_obj["last_verified_utc"] = datetime.now(timezone.utc).isoformat()
write_json(SANDBOX_MANIFEST, sandbox_manifest_obj)

summary.update({
    "sandbox_sha256": sandbox_hash,
    "integrated_sha256": integrated_hash,
    "splits_sha256": splits_hash,
    "unlabelled_candidates_sha256": unlabelled_hash,
    "integrated_records": len(disk_integrated),
    "scienceworld_labelled_records": len(sw_labelled),
    "scienceworld_unlabelled_candidates": len(disk_unlabelled),
})
write_json(SUMMARY_JSON, summary)

# Re-open metadata and verify consistency.
saved_summary = read_json(SUMMARY_JSON)
saved_sandbox_manifest = read_json(SANDBOX_MANIFEST)
saved_splits = read_json(SPLITS_JSON)
assert saved_summary["sandbox_sha256"] == sandbox_hash
assert saved_summary["integrated_sha256"] == integrated_hash
assert saved_summary["splits_sha256"] == splits_hash
assert saved_sandbox_manifest["dataset_sha256"] == sandbox_hash
assert saved_splits["dataset_sha256"] == integrated_hash

checklist = {
    "sandbox_record_count_2000": len(disk_sandbox) == 2000,
    "sandbox_ids_unique": len(sandbox_ids) == 2000,
    "independent_reference_audit_passed": len(audit_failures) == 0,
    "minimal_pair_validation_passed": len(pair_failures) == 0,
    "sandbox_schema_validation_passed": len(sandbox_errors) == 0,
    "integrated_schema_validation_passed": len(integrated_errors) == 0,
    "integrated_record_count_2001": len(disk_integrated) == 2001,
    "scienceworld_labelled_record_preserved": len(sw_labelled) == 1,
    "orange_candidate_kept_unlabelled": disk_unlabelled[0]["label"] == "unlabelled_candidate",
    "candidate_excluded_from_integrated_dataset": disk_unlabelled[0]["id"] not in set(integrated_ids),
    "all_grouped_splits_revalidated": True,
    "sandbox_manifest_hash_matches": saved_sandbox_manifest["dataset_sha256"] == sandbox_hash,
    "summary_hashes_match": (
        saved_summary["sandbox_sha256"] == sandbox_hash
        and saved_summary["integrated_sha256"] == integrated_hash
        and saved_summary["splits_sha256"] == splits_hash
    ),
    "split_dataset_hash_matches": saved_splits["dataset_sha256"] == integrated_hash,
}

print("=" * 72)
print("NB4 FINAL INTEGRITY REPORT")
print("=" * 72)
print("Sandbox records:", len(disk_sandbox))
print("Integrated labelled records:", len(disk_integrated))
print("Unlabelled candidates:", len(disk_unlabelled))
print("Environment counts:", dict(Counter(row["environment"] for row in disk_integrated)))
print("Integrated classes:", dict(Counter(row["label"] for row in disk_integrated)))
print("Sandbox SHA-256:", sandbox_hash)
print("Integrated SHA-256:", integrated_hash)
print("Splits SHA-256:", splits_hash)
print("Unlabelled SHA-256:", unlabelled_hash)
print("CHECKLIST:\n", json.dumps(checklist, indent=2))
assert all(checklist.values()), "One or more NB4 final integrity checks failed."
print("\nNB4 FINAL INTEGRITY CHECKS PASSED.")


NB4 FINAL INTEGRITY REPORT
Sandbox records: 2000
Integrated labelled records: 2001
Unlabelled candidates: 1
Environment counts: {'python_sandbox': 2000, 'scienceworld': 1}
Integrated classes: {'reversible': 751, 'irreversible': 1000, 'partially_reversible': 250}
Sandbox SHA-256: 08726639568dc289313e00707349cea7a270e513a379e4369de7ec1959a2a1d8
Integrated SHA-256: 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd
Splits SHA-256: 6033777a50dd930ef20942c61b1fad485962d9e0f980bde62023bcb84ef90b6d
Unlabelled SHA-256: a5b5486f9d74bd06a5357cecac96713783b3971958c9697ef47ca98dc9926894
CHECKLIST:
 {
  "sandbox_record_count_2000": true,
  "sandbox_ids_unique": true,
  "independent_reference_audit_passed": true,
  "minimal_pair_validation_passed": true,
  "sandbox_schema_validation_passed": true,
  "integrated_schema_validation_passed": true,
  "integrated_record_count_2001": true,
  "scienceworld_labelled_record_preserved": true,
  "orange_candidate_kept_unlabelled": true,
  "candida

## 14. Immutable timestamped Drive backups

In [15]:
# ============================================================
# SECTION 14 — IMMUTABLE TIMESTAMPED DRIVE BACKUPS
# ============================================================

backup_files = [
    SANDBOX_JSONL,
    SANDBOX_MANIFEST,
    INTEGRATED_JSONL,
    UNLABELLED_JSONL,
    SPLITS_JSON,
    SUMMARY_JSON,
    DATA_CARD,
]
for path in backup_files:
    if not path.exists():
        raise FileNotFoundError(f"Required backup artifact missing: {path}")

# Build the reproducibility manifest after final data and metadata hashes settle.
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "generator_version": GENERATOR_VERSION,
    "seed": SEED,
    "sandbox_records": len(disk_sandbox),
    "integrated_records": len(disk_integrated),
    "unlabelled_candidate_records": len(disk_unlabelled),
    "sandbox_sha256": sha256_file(SANDBOX_JSONL),
    "sandbox_manifest_sha256": sha256_file(SANDBOX_MANIFEST),
    "integrated_sha256": sha256_file(INTEGRATED_JSONL),
    "unlabelled_sha256": sha256_file(UNLABELLED_JSONL),
    "splits_sha256": sha256_file(SPLITS_JSON),
    "summary_sha256": sha256_file(SUMMARY_JSON),
    "data_card_sha256": sha256_file(DATA_CARD),
    "nb3_source_sha256": sha256_file(scienceworld_path),
    "checklist": checklist,
}
MANIFEST_JSON = REPORT_DIR / "NB4_reproducibility_manifest_v2.json"
write_json(MANIFEST_JSON, manifest)
backup_files.append(MANIFEST_JSON)

# Timestamp with microseconds avoids collisions and preserves earlier snapshots.
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
snapshot_name = f"NB4_v2_snapshot_{timestamp}"
backup_results = []

for root in (DRIVE_ROOT, BACKUP_ROOT):
    target = root / snapshot_name
    target.mkdir(parents=True, exist_ok=False)
    verified_files = []

    for src in backup_files:
        dst = target / src.name
        shutil.copy2(src, dst)
        source_hash = sha256_file(src)
        destination_hash = sha256_file(dst)
        ok = (
            src.stat().st_size == dst.stat().st_size
            and source_hash == destination_hash
        )
        verified_files.append({
            "file": src.name,
            "bytes": dst.stat().st_size,
            "source_sha256": source_hash,
            "backup_sha256": destination_hash,
            "verified": ok,
        })
        if not ok:
            raise IOError(f"Backup verification failed: {dst}")

    backup_results.append({
        "snapshot": str(target),
        "files": verified_files,
        "all_files_verified": all(item["verified"] for item in verified_files),
    })

# Cross-check corresponding files in both Drive roots.
main_snapshot = DRIVE_ROOT / snapshot_name
backup_snapshot = BACKUP_ROOT / snapshot_name
for src in backup_files:
    assert sha256_file(main_snapshot / src.name) == sha256_file(backup_snapshot / src.name), (
        f"Main and independent backup differ: {src.name}"
    )

backup_manifest = {
    "snapshot_name": snapshot_name,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "snapshots": backup_results,
    "cross_root_hashes_match": True,
}
LOCAL_BACKUP_MANIFEST = REPORT_DIR / f"{snapshot_name}_manifest.json"
write_json(LOCAL_BACKUP_MANIFEST, backup_manifest)

print(json.dumps(backup_manifest, indent=2))
print("NB4 snapshot:", snapshot_name)
print("Main Drive:", main_snapshot)
print("Independent backup:", backup_snapshot)
print("All copied files verified:", True)


{
  "snapshot_name": "NB4_v2_snapshot_20261123T130007318638Z",
  "created_utc": "2026-11-23T13:00:07.721760+00:00",
  "snapshots": [
    {
      "snapshot": "/content/drive/MyDrive/ir/NB4_v2_snapshot_20261123T130007318638Z",
      "files": [
        {
          "file": "sandbox_pairs_v2.jsonl",
          "bytes": 2243505,
          "source_sha256": "08726639568dc289313e00707349cea7a270e513a379e4369de7ec1959a2a1d8",
          "backup_sha256": "08726639568dc289313e00707349cea7a270e513a379e4369de7ec1959a2a1d8",
          "verified": true
        },
        {
          "file": "sandbox_manifest_v2.json",
          "bytes": 699,
          "source_sha256": "5dd520af9acb1f6d57cd795d8f52f2d545677236a13ff6507063c8955953cba2",
          "backup_sha256": "5dd520af9acb1f6d57cd795d8f52f2d545677236a13ff6507063c8955953cba2",
          "verified": true
        },
        {
          "file": "integrated_dataset_v2.jsonl",
          "bytes": 2249352,
          "source_sha256": "0d890a39de3953e1a4d8f03d0

## 15. Completion checklist

In [16]:
# ============================================================
# SECTION 15 — NB4 COMPLETION STATUS
# ============================================================

assert all(checklist.values()), "NB4 cannot be marked complete."
assert backup_manifest["cross_root_hashes_match"] is True

print("=" * 72)
print("NB4 v2 COMPLETE")
print("=" * 72)
print("Sandbox records:", len(disk_sandbox))
print("Integrated labelled records:", len(disk_integrated))
print("Unlabelled candidates:", len(disk_unlabelled))
print("Minimal pairs:", len(pair_groups))
print("Split seeds:", list(splits.keys()))
print("All integrity checks passed:", all(checklist.values()))
print("All backup copies verified:", backup_manifest["cross_root_hashes_match"])
print("Snapshot:", snapshot_name)
print("Main Drive:", DRIVE_ROOT / snapshot_name)
print("Independent backup:", BACKUP_ROOT / snapshot_name)


NB4 v2 COMPLETE
Sandbox records: 2000
Integrated labelled records: 2001
Unlabelled candidates: 1
Minimal pairs: 1000
Split seeds: ['0', '1', '2']
All integrity checks passed: True
All backup copies verified: True
Snapshot: NB4_v2_snapshot_20261123T130007318638Z
Main Drive: /content/drive/MyDrive/ir/NB4_v2_snapshot_20261123T130007318638Z
Independent backup: /content/drive/MyDrive/ir_final/NB4_v2_snapshot_20261123T130007318638Z
